# import

In [ ]:
from utils.recbole_train_test import *
from utils.plot_utils import *
from utils.model_utils import get_trainer
from utils.custom_trainer import *

In [ ]:
# MODEL_VERSIONS = ['_pt1', '_pt2', '_pt3', '_pt4']
freq=6 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]
MODEL_VERSIONS = d_keys[:duration]


Ks = [1, 10, 20]
VM_K = Ks[2] # valid metric k, also used in heatmap matrix
VALID_METRIC = 'Recall@'+str(VM_K)
SEED = 2020
USE_GPU = False
SHOW_PROGRESS = False
SAVE_DATASET = False

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

METRICS = ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision', 'GiniIndex', 'TailPercentage']

FILENAME_VERSION = '_ET_LS.t_UD_SF_TO_UM.100'

# Experiment Goodreads - ET RD LS.t UD SF TO UM.100
executed train, random drift, leave one out train sampled, uniform neg sample training distribution, shuffle false, time ordering, uniform mode


In [ ]:
save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df', 
                                        '') # NPT_NT_RD.50
BENCHMARK_FILENAMES = ['train', 'valid', 'test']
base_dataset_name = base_filename+'_'+specs_str

# BPR

In [ ]:
model_name = 'BPR'

for part in MODEL_VERSIONS:
    print('\n\n'+part)
    current_dataset_name = base_dataset_name+part

    # pre trained model 
    model_checkpoint_ver = model_name+'-Mar-'
    model_checkpoint_dir = save_path+base_filename+part
    model_checkpoint_file = model_checkpoint_dir+'/'+model_checkpoint_ver+'.pth'


    parameter_dict = {  'dataset': current_dataset_name+'.inter',
                        'use_gpu':USE_GPU,

                        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
                        'seed':SEED,
                        'state':'ERROR',
                        'data_path': save_path,
                        'save_dataset':SAVE_DATASET, #(bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
                        'checkpoint_dir':save_path+base_dataset_name,
                        'show_progress': SHOW_PROGRESS,
                        'shuffle': SHUFFLE,

                        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
                        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
                        # 'user_inter_num_interval':'[1,inf)',
                        'benchmark_filename': BENCHMARK_FILENAMES,
                        
                        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
                        # 'train_neg_sample_args': TRAIN_NEG_SAMPLE_ARGS,
                        

                        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
                        'eval_args': EVAL_ARGS,
                        'metrics': METRICS, 
                        'topk':Ks,
                        'valid_metric':VALID_METRIC          
                    }

  
    

    # current_config,current_logger,current_dataset,current_train_data, current_valid_data, current_test_data
    current_config,\
        current_logger, _,\
            current_train_data,\
                current_valid_data,\
                    current_test_data = setup_config_and_dataset(model_name,
                                                                current_dataset_name,
                                                                parameter_dict)

    # model loading and initialization
    current_model = BPR(current_config, current_train_data.dataset).to(current_config['device'])
    current_logger.info(current_model)


    # trainer loading and initialization
    trainer = get_trainer(current_config['MODEL_TYPE'], current_config['model'])(current_config, current_model)


    # model training
    best_valid_score, best_valid_result = trainer.fit(current_train_data, current_valid_data)
    print('\n\nTraining best results')
    print('best_valid_score: ', best_valid_score)
    print('best_valid_result: ', best_valid_result)


    # main diagonal eval
    test_result = evaluate(trainer, current_test_data)
    save_evaluation_results(test_result, 
                            parameter_dict['checkpoint_dir'], 
                            get_evaluation_results_filename(current_config['model'], 
                                                            current_dataset_name, 
                                                            part, 
                                                            FILENAME_VERSION))


    
    test_full_data_sections = get_test_full_data_sections_with_names(model_version=part,
                                            base_dataset_name=base_dataset_name,
                                            models_versions=MODEL_VERSIONS)[:-1]
    print(test_full_data_sections)
    

    # evaluate in all testsets
    for testset_name in test_full_data_sections:

        _,_,\
            _,trainset,_,\
                testset = setup_config_and_dataset(model_name,
                                                    testset_name,
                                                    parameter_dict)

        # When calculate ItemCoverage metrics, we need to run this code for set item_nums in eval_collector.
        trainer.eval_collector.data_collect(trainset)
        
        test_result = evaluate(trainer, testset)
        save_evaluation_results(test_result, 
                                parameter_dict['checkpoint_dir'], 
                                get_evaluation_results_filename(current_config['model'], 
                                                                current_dataset_name, 
                                                                testset_name, 
                                                                FILENAME_VERSION))

### recall heatmap

In [ ]:
model_name = 'BPR'

results_matrix = get_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(VM_K),
                                    filename_version=FILENAME_VERSION,
                                    part_shift_incl=False,
                                    test_full_data_sec=True)

recall_heatmap(results_matrix, 
               round_point=4, 
               title='BPR - Goodreads - 50\% Random Drift', 
               filepath='images/goodreads/'+base_dataset_name+FILENAME_VERSION+'_'+model_name)

# NeuMF

# Pop

# ItemKNN